In [ ]:
!nvidia-smi

In [ ]:
import gc
import os
import re

import cv2
import lpips
import numpy as np
import pandas as pd
import piq  # <- para brisque()
import pyiqa  # <- para NIQE
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from piq import fsim, psnr, ssim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import save_image
from tqdm import tqdm
gc.collect()
torch.cuda.empty_cache()
# My libraries
#%run src/Models/Simple_GAN.ipynb
#%run src/Models/GAN_1.ipynb
%run src/Models/GAN_2.ipynb
#%run src/Models/GAN_3.ipynb
#%run src/Models/GAN_2_Lite.ipynb
%run src/LOLDataset.ipynb
%run src/FiveKPairs.ipynb

In [ ]:
DATASETS = {
    "LOLv1": {
        "train": {
            "NORMAL": "../datasets/lol_dataset/our485/high/",
            "LOW":    "../datasets/lol_dataset/our485/low/",
        },
        "test": {
            "LOW":    "../datasets/lol_dataset/eval15/low/",
            "NORMAL": "../datasets/lol_dataset/eval15/high/",
        },
        "datasetclass":"LowLightDataset"
    },

    "LOL-v2-R": {
        "train": {
            "NORMAL": "../datasets/LOL-v2/Real_captured/Train/Normal/",
            "LOW":    "../datasets/LOL-v2/Real_captured/Train/Low/",
        },
        "test": {
            "LOW":    "../datasets/LOL-v2/Real_captured/Test/Low/",
            "NORMAL": "../datasets/LOL-v2/Real_captured/Test/Normal/",
        },
        "datasetclass":"LOLv2Dataset"
    },

    "LOL-v2-S": {
        "train": {
            "NORMAL": "../datasets/LOL-v2/Synthetic/Train/Normal/",
            "LOW":    "../datasets/LOL-v2/Synthetic/Train/Low/",
        },
        "test": {
            "LOW":    "../datasets/LOL-v2/Synthetic/Test/Low/",
            "NORMAL": "../datasets/LOL-v2/Synthetic/Test/Normal/",
        },
        "datasetclass":"LowLightDataset"

    },
    "LoLI-Street": {
        "train": {
            "NORMAL": "../datasets/LoLI-Street/Train/high/",
            "LOW":    "../datasets/LoLI-Street/Train/low/",
        },
        "test": {
            "LOW":    "../datasets/LoLI-Street/Val/low/",
            "NORMAL": "../datasets/LoLI-Street/Val/high/",
        },
        "datasetclass":"LowLightDataset"

    },
     "FiveKPairs": {
        "train": {
            "NORMAL": "../datasets/FiveKpairs/training/GT_IMAGES/",
            "LOW":    "../datasets/FiveKpairs/training/INPUT_IMAGES/",
        },
        "test": {
            "NORMAL": "../datasets/FiveKpairs/validation/GT_IMAGES/",
            "LOW":    "../datasets/FiveKpairs/validation/INPUT_IMAGES/",
        },
        "datasetclass":"FiveKPairs"
    },
    
}

In [ ]:
Dataset_Names = ['LOLv1','LOL-v2-R', 'LOL-v2-S', "LoLI-Street", "FiveKPairs" ]
Dataset_Name  = Dataset_Names[0]
Dataset       = DATASETS[Dataset_Name]
TRAIN_LOW, TRAIN_NORMAL = Dataset['train']['LOW'] , Dataset['train']['NORMAL']
TEST_LOW, TEST_NORMAL   = Dataset['test']['LOW'] , Dataset['test']['NORMAL']

print(Dataset_Name)

In [ ]:
import inspect
import time 

Models = {
    #"SimpleGAN": {"G": Generator,        "D": Discriminator},
    #"GAN1":      {"G": ResUNetLLIE,      "D": PatchDiscriminatorSN},
    "GAN2":      {"G": ResUNetLLIEV2,    "D": PatchDiscriminatorSNCondV2},
    #"GAN2_Lite": {"G": ResUNetLLIE_Lite, "D": PatchDiscriminatorSNCond_Lite},
}

Model_Name = "GAN2"
print("Model name:", Model_Name)
Model = Models[Model_Name]

In [ ]:

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

def model_size_mb(model):
    param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024**2)

def _try_flops_fvcore(model, inputs_tuple):
    try:
        from fvcore.nn import FlopCountAnalysis
        flops = FlopCountAnalysis(model, inputs_tuple).total()
        return float(flops)
    except Exception:
        return None

def _try_flops_thop(model, inputs_tuple):
    try:
        from thop import profile
        flops, _ = profile(model, inputs=inputs_tuple, verbose=False)
        return float(flops)
    except Exception:
        return None

def get_flops(model, inputs_tuple):
    # 1) fvcore (si existe) 2) thop (si existe) 3) None
    fl = _try_flops_fvcore(model, inputs_tuple)
    if fl is not None:
        return fl
    fl = _try_flops_thop(model, inputs_tuple)
    return fl  # puede ser None

@torch.no_grad()
def measure_latency_ms(model, inputs_tuple, device, iters=100, warmup=30):
    model.eval()

    # warmup
    for _ in range(warmup):
        _ = model(*inputs_tuple)

    if device.type == "cuda":
        torch.cuda.synchronize()
        starter = torch.cuda.Event(enable_timing=True)
        ender   = torch.cuda.Event(enable_timing=True)

        starter.record()
        for _ in range(iters):
            _ = model(*inputs_tuple)
        ender.record()

        torch.cuda.synchronize()
        return starter.elapsed_time(ender) / iters  # ms/iter
    else:
        t0 = time.perf_counter()
        for _ in range(iters):
            _ = model(*inputs_tuple)
        t1 = time.perf_counter()
        return (t1 - t0) * 1000.0 / iters  # ms/iter

def guess_inputs(model, H=400, W=600,  device=None):
    if device is None:
        device = next(model.parameters()).device

    sig = inspect.signature(type(model).forward)
    params = list(sig.parameters.values())
    params = params[1:]
    required = [p for p in params
                if p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)
                and p.default is p.empty]
    n = len(required)

    def try_run(inputs):
        try:
            _ = model(*inputs)
            return True
        except Exception:
            return False

    if n <= 1:
        x3 = torch.randn(1, 3, H, W, device=device)
        if try_run((x3,)):
            return (x3,)
        x6 = torch.randn(1, 6, H, W, device=device)
        if try_run((x6,)):
            return (x6,)
        # último intento: 1ch (por si tu red trabaja grayscale)
        x1 = torch.randn(1, 1, H, W, device=device)
        return (x1,)

    if n == 2:
        x = torch.randn(1, 3, H, W, device=device)
        y = torch.randn(1, 3, H, W, device=device)
        return (x, y)

    # n >= 3
    x = torch.randn(1, 3, H, W, device=device)
    y = torch.randn(1, 3, H, W, device=device)
    z = torch.zeros(1, dtype=torch.long, device=device)  # etiqueta dummy
    return (x, y, z)

def analyze_one(model_cls, device, H=400, W=600, ctor_kwargs=None):
    ctor_kwargs = ctor_kwargs or {}
    model = model_cls(**ctor_kwargs).to(device).eval()

    inputs = guess_inputs(model, H=H, W=W, device=device)

    total_p, train_p = count_params(model)
    size_mb = model_size_mb(model)
    flops = get_flops(model, inputs)
    latency_ms = measure_latency_ms(model, inputs, device=device, iters=100, warmup=30)

    return {
        "params_total": total_p,
        "params_trainable": train_p,
        "size_mb": size_mb,
        "flops": flops,             
        "latency_ms": latency_ms,
        "input_shapes": [tuple(t.shape) if torch.is_tensor(t) else str(type(t)) for t in inputs]
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True if device.type == "cuda" else False
print("Device:", device)

results = {}
for role in ["G", "D"]:
    results[role] = analyze_one(Model[role], device=device, H=400, W=600)

print(f"\n=== {Model_Name} ===")
for role in ["G", "D"]:
    r = results[role]
    print(f"\n[{role}]")
    print(" input_shapes      :", r["input_shapes"])
    print(" params_total      :", f'{r["params_total"]:,}')
    print(" params_trainable  :", f'{r["params_trainable"]:,}')
    print(" size_mb           :", f'{r["size_mb"]:.2f} MB')
    print(" flops             :", "None (instala fvcore o thop)" if r["flops"] is None else f'{r["flops"]:.3e}')
    print(" latency_ms        :", f'{r["latency_ms"]:.2f} ms/iter')

# all_rows = []
# for name, md in Models.items():
#     for role in ["G", "D"]:
#         r = analyze_one(md[role], device=device, H=256, W=256)
#         all_rows.append({
#             "model": name, "net": role,
#             "params_total": r["params_total"],
#             "size_mb": r["size_mb"],
#             "flops": r["flops"],
#             "latency_ms": r["latency_ms"],
#             "input_shapes": str(r["input_shapes"]),
#         })
# import pandas as pd
# df = pd.DataFrame(all_rows)
# print(df)
# df.to_csv("model_stats.csv", index=False)

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((400, 600)),
    transforms.ToTensor(),
])

device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size    = 2
epochs        = 200
lr            = 1e-4
NORMAL        = TRAIN_NORMAL 
LOW           = TRAIN_LOW
dataset       = eval(Dataset["datasetclass"])(low_light_dir=LOW, normal_dir=NORMAL, transform=transform)#FiveKPairs(input_dir=TRAIN_LOW, gt_dir=TRAIN_NORMAL, mode="low", strict=True,return_tag=False, transform=transform)
dataloader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
Generator     = Model["G"]#ResUNetLLIE_Lite          #ResUNetLLIEV2
Discriminator = Model["D"]#PatchDiscriminatorSNCond_Lite #PatchDiscriminatorSN #PatchDiscriminatorSNCondV2#
generator     = Generator(base=32, n_res=6).to(device)
discriminator = Discriminator(base=64).to(device)

adversarial_loss = nn.BCEWithLogitsLoss()#nn.BCELoss()
pixelwise_loss   = nn.L1Loss()
optimizer_G      = optim.AdamW(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D      = optim.AdamW(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Training 

In [ ]:
import os
import torch
from tqdm import tqdm
from torchvision.utils import save_image

def D_forward(discriminator, low_light, img, D_COND: bool):
    """
    - Si D_COND=False: D(img)
    - Si D_COND=True : D(low_light, img)  (Pix2Pix-style)
    """
    return discriminator(low_light, img) if D_COND else discriminator(img)

os.makedirs("results", exist_ok=True)
g_model_name      = "generator_"+Model_Name+"_last.pth"
d_model_name      = "discriminator"+Model_Name+"_last.pth"
g_best_model_name = "generator_"+Model_Name+"_best.pth"
d_best_model_name = "discriminator_"+Model_Name+"_best.pth"

best_g_loss = float("inf")
best_epoch  = -1
D_COND      = True if ( Model_Name == 'GAN2' or Model_Name == 'GAN2_Lite') else False

fixed_low, fixed_normal = next(iter(dataloader))
fixed_low               = fixed_low.to(device)

for epoch in range(epochs):
    generator.train()
    discriminator.train()

    g_loss_sum = 0.0
    d_loss_sum = 0.0
    n_batches  = 0

    for i, (low_light, normal) in enumerate(tqdm(dataloader)):
        low_light   = low_light.to(device)
        normal      = normal.to(device)
        fake_normal = generator(low_light)
        optimizer_D.zero_grad()

        pred_real = D_forward(discriminator, low_light, normal, D_COND)
        loss_real = adversarial_loss(pred_real, torch.ones_like(pred_real))
        
        pred_fake = D_forward(discriminator, low_light, fake_normal.detach(), D_COND)
        loss_fake = adversarial_loss(pred_fake, torch.zeros_like(pred_fake))

        d_loss = 0.5 * (loss_real + loss_fake)
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        pred_fake_for_g  = D_forward(discriminator, low_light, fake_normal, D_COND)
        g_loss_adv       = adversarial_loss(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
        g_loss_pixelwise = pixelwise_loss(fake_normal, normal)

        g_loss = g_loss_adv + 100.0 * g_loss_pixelwise

        g_loss.backward()
        optimizer_G.step()

        g_loss_sum += g_loss.item()
        d_loss_sum += d_loss.item()
        n_batches  += 1

    avg_g = g_loss_sum / max(1, n_batches)
    avg_d = d_loss_sum / max(1, n_batches)

    print(f"Epoch {epoch+1}/{epochs} | D_Loss: {avg_d:.4f} | G_Loss: {avg_g:.4f}")

    generator.eval()
    with torch.no_grad():
        sample_fake = generator(fixed_low)
    save_image(sample_fake, f"results/epoch_{epoch+1}.png")
    
    if avg_g < best_g_loss:
        best_g_loss = avg_g
        best_epoch  = epoch + 1
        torch.save(generator.state_dict(), g_best_model_name)
        torch.save(discriminator.state_dict(), d_best_model_name)
        print(f"✅ Best epoch {best_epoch} with G_Loss={best_g_loss:.4f}")

    #torch.save(generator.state_dict(), g_model_name)
    #torch.save(discriminator.state_dict(), d_model_name)

print(f"\n🏁 Best epoch: {best_epoch} | Best G_Loss: {best_g_loss:.4f}")
print(f"Best: {g_best_model_name}, {d_best_model_name}")
print(f"Last: {g_model_name}, {d_model_name}")

g_model_name = g_best_model_name
d_model_name = d_best_model_name

# <center style="font-size:30px;">Testing </center>

In [ ]:
import os
import re
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from piq import psnr, ssim, fsim
import piq  # <- para brisque()
import lpips
import pyiqa  # <- para NIQE

from PIL import Image
import pandas as pd
import numpy as np

def resolve_high_name(dataset_key: str, low_name: str) -> str:
    # LOLv1: mismo nombre
    if dataset_key == "LOLv1" or dataset_key == "LoLI-Street":
        return low_name

    # LOL-v2-R / LOL-v2-S: low0001.png -> normal0001.png
    if dataset_key.startswith("LOL-v2"):
        m = re.match(r"(?i)^low(\d+)\.(png|jpg|jpeg|bmp|tif|tiff)$", low_name)
        if not m:
            # si ya viene como normal#### o es otro formato, intenta mismo nombre
            return low_name
        return f"normal{m.group(1)}.{m.group(2)}"

    # fallback
    return low_name

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using  device:", device)


dataset_key = Dataset_Name #"LOLv1"  # <-- "LOLv1" | "LOL-v2-R" | "LOL-v2-S"
split       = "test"            # <-- "train" o "test"

dataset_name = f"{dataset_key} ({split})"

Dataset = DATASETS[dataset_key]  

folder_low  = TEST_LOW    
folder_high = TEST_NORMAL 

safe_ds = dataset_key.replace("-", "_")
out_dir = f"./results_{split}_enhanced_{safe_ds}"
os.makedirs(out_dir, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

transform = transforms.ToTensor()

generator  = Generator(base=32, n_res=6).to(device)
state_dict = torch.load(g_best_model_name, map_location=device)
generator.load_state_dict(state_dict)
generator.eval()

lpips_fn = lpips.LPIPS(net='alex').to(device).eval()
niqe_fn  = pyiqa.create_metric('niqe', device=device).eval()

image_names = sorted([
    f for f in os.listdir(folder_low)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
])

psnr_list  = []
ssim_list  = []
mse_list   = []
fsim_list  = []
lpips_list = []

niqe_list    = []
brisque_list = []

for img_name in image_names:
    path_low = os.path.join(folder_low, img_name)

    high_name = resolve_high_name(dataset_key, img_name)
    path_high = os.path.join(folder_high, high_name)

    # print("GT:", path_high)

    if not os.path.exists(path_high):
        print(f"⚠️ GT for : {img_name} -> {high_name}")
        continue

    low_pil  = Image.open(path_low).convert("RGB")
    high_pil = Image.open(path_high).convert("RGB")

    input_tensor = transform(low_pil).unsqueeze(0).to(device)   # (1, C, H, W)
    high_tensor  = transform(high_pil).unsqueeze(0).to(device)  # (1, C, H, W)

    with torch.no_grad():
        enhanced = generator(input_tensor)

    enhanced = torch.clamp(enhanced, 0.0, 1.0)

    enhanced_for_metrics = enhanced.cpu()
    high_for_metrics     = high_tensor.cpu()
    data_range = 1.0

    psnr_value = psnr(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    ssim_value = ssim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    fsim_value = fsim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    mse_value  = F.mse_loss(enhanced_for_metrics, high_for_metrics).item()

    psnr_list.append(psnr_value)
    ssim_list.append(ssim_value)
    mse_list.append(mse_value)
    fsim_list.append(fsim_value)

    # ---------------- LPIPS ----------------
    enhanced_lpips = enhanced * 2.0 - 1.0
    high_lpips     = high_tensor * 2.0 - 1.0
    with torch.no_grad():
        lpips_value = lpips_fn(enhanced_lpips, high_lpips).mean().item()
    lpips_list.append(lpips_value)

    # ---------------- NO-REFERENCE (NIQE/BRISQUE) ----------------
    with torch.no_grad():
        niqe_value = niqe_fn(enhanced).item()
    niqe_list.append(niqe_value)

    brisque_value = piq.brisque(enhanced_for_metrics, data_range=data_range).item()
    brisque_list.append(brisque_value)

    save_path = os.path.join(out_dir, img_name)  # guarda con el nombre LOW
    save_image(enhanced, save_path)

if len(psnr_list) == 0:
    raise RuntimeError("No GT .")

def mean_std(x):
    x = np.array(x, dtype=np.float64)
    return x.mean(), x.std(ddof=1)

mean_psnr, std_psnr = mean_std(psnr_list)
mean_ssim, std_ssim = mean_std(ssim_list)
mean_mse,  std_mse  = mean_std(mse_list)
mean_fsim, std_fsim = mean_std(fsim_list)
mean_lpips, std_lpips = mean_std(lpips_list)

mean_niqe, std_niqe = mean_std(niqe_list)
mean_brisque, std_brisque = mean_std(brisque_list)

print(f"Dataset: {dataset_name}")
print(f"images: {len(psnr_list)}")
print(f"PSNR mean    : {mean_psnr:.2f} ± {std_psnr:.2f} dB")
print(f"SSIM mean    : {mean_ssim:.4f} ± {std_ssim:.4f}")
print(f"MSE mean     : {mean_mse:.6f} ± {std_mse:.6f}")
print(f"FSIM mean    : {mean_fsim:.4f} ± {std_fsim:.4f}")
print(f"LPIPS mean   : {mean_lpips:.4f} ± {std_lpips:.4f}  (↓ mejor)")
print(f"NIQE mean    : {mean_niqe:.4f} ± {std_niqe:.4f}    (↓ mejor)")
print(f"BRISQUE mean : {mean_brisque:.4f} ± {std_brisque:.4f} (↓ mejor)")


summary_csv = "llie_resultados_globales.csv"

summary_row = {
    "Dataset": dataset_name,
    "DatasetKey": dataset_key,
    "Split": split,
    "Num_Images": len(psnr_list),
    "PSNR_mean": mean_psnr, "PSNR_std": std_psnr,
    "SSIM_mean": mean_ssim, "SSIM_std": std_ssim,
    "MSE_mean": mean_mse, "MSE_std": std_mse,
    "FSIM_mean": mean_fsim, "FSIM_std": std_fsim,
    "LPIPS_mean": mean_lpips, "LPIPS_std": std_lpips,
    "NIQE_mean": mean_niqe, "NIQE_std": std_niqe,
    "BRISQUE_mean": mean_brisque, "BRISQUE_std": std_brisque,
}

df_summary = pd.DataFrame([summary_row])

if not os.path.exists(summary_csv):
    df_summary.to_csv(summary_csv, index=False)
else:
    df_summary.to_csv(summary_csv, mode='a', header=False, index=False)

print(f"\nSaved in: {summary_csv}")
print(f"Saved in: {out_dir}")


In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from piq import psnr, ssim, fsim
import piq  # <- para brisque()
import lpips
import pyiqa  # <- para NIQE

from PIL import Image
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# ------------------------------------------------------------------
# 1. Nombre del dataset y rutas
# ------------------------------------------------------------------
dataset_name = "LOL Test"  

folder_low  = Dataset['test']['LOW']#'../datasets/lol_dataset/eval15/low/' #LOL-v2/Real_captured/Test/Low/' #lol_dataset/eval15/low/'   # imágenes low-light (input del modelo)
folder_high = Dataset['test']['NORMAL']#'../datasets/lol_dataset/eval15/high/'#LOL-v2/Real_captured/Test/Normal/'#lol_dataset/eval15/high/'  # ground truth
out_dir     = "./results_eval15_enhanced_LOLv2-R"         # donde se guardarán las mejoradas
os.makedirs(out_dir, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

transform = transforms.ToTensor()

generator = Generator().to(device)
state_dict = torch.load(g_model_name, map_location=device)
generator.load_state_dict(state_dict)
generator.eval()


lpips_fn = lpips.LPIPS(net='alex').to(device).eval()

niqe_fn = pyiqa.create_metric('niqe', device=device).eval()  # NIQE (↓ mejor)


image_names = sorted([
    f for f in os.listdir(folder_low)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
])

print(f"Se encontraron {len(image_names)} imágenes low para evaluar.")


psnr_list  = []
ssim_list  = []
mse_list   = []
fsim_list  = []
lpips_list = []

niqe_list    = []
brisque_list = []

for img_name in image_names:
    path_low  = os.path.join(folder_low, img_name)
    path_high = os.path.join(folder_high, img_name)
    print(path_high)
    if not os.path.exists(path_high):
        print(f"⚠️ No se encontró la GT para: {img_name}")
        continue

    # ---------------- Cargar imágenes como PIL ----------------
    low_pil  = Image.open(path_low).convert("RGB")
    high_pil = Image.open(path_high).convert("RGB")

    # ---------------- Aplicar transform ----------------
    input_tensor = transform(low_pil).unsqueeze(0).to(device)   # (1, C, H, W), [0,1]
    high_tensor  = transform(high_pil).unsqueeze(0).to(device)  # (1, C, H, W), [0,1]

    # ---------------- Inferencia ----------------
    with torch.no_grad():
        enhanced = generator(input_tensor)  # salida del modelo

    enhanced = torch.clamp(enhanced, 0.0, 1.0)

    # ---------------- Métricas (PSNR/SSIM/FSIM/MSE) ----------------
    enhanced_for_metrics = enhanced.cpu()
    high_for_metrics     = high_tensor.cpu()
    data_range = 1.0

    psnr_value = psnr(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    ssim_value = ssim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    fsim_value = fsim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    mse_value  = F.mse_loss(enhanced_for_metrics, high_for_metrics).item()

    psnr_list.append(psnr_value)
    ssim_list.append(ssim_value)
    mse_list.append(mse_value)
    fsim_list.append(fsim_value)

    # ---------------- LPIPS ----------------
    enhanced_lpips = enhanced * 2.0 - 1.0
    high_lpips     = high_tensor * 2.0 - 1.0
    with torch.no_grad():
        lpips_value = lpips_fn(enhanced_lpips, high_lpips).mean().item()
    lpips_list.append(lpips_value)

    # ---------------- NO-REFERENCE (NIQE/BRISQUE) ----------------
    # NIQE con pyiqa (↓ mejor)
    with torch.no_grad():
        niqe_value = niqe_fn(enhanced).item()
    niqe_list.append(niqe_value)

    # BRISQUE con piq (↓ mejor)
    brisque_value = piq.brisque(enhanced_for_metrics, data_range=data_range).item()
    brisque_list.append(brisque_value)

    # ---------------- Guardar salida visual ----------------
    save_path = os.path.join(out_dir, img_name)
    save_image(enhanced, save_path)

# ------------------------------------------------------------------
# 6. Métricas globales (media y desviación estándar)
# ------------------------------------------------------------------
if len(psnr_list) == 0:
    raise RuntimeError("No se calcularon métricas: revisa que haya imágenes y GT correspondientes.")

def mean_std(x):
    x = np.array(x, dtype=np.float64)
    return x.mean(), x.std(ddof=1)

mean_psnr, std_psnr = mean_std(psnr_list)
mean_ssim, std_ssim = mean_std(ssim_list)
mean_mse,  std_mse  = mean_std(mse_list)
mean_fsim, std_fsim = mean_std(fsim_list)
mean_lpips, std_lpips = mean_std(lpips_list)

mean_niqe, std_niqe = mean_std(niqe_list)
mean_brisque, std_brisque = mean_std(brisque_list)

print("\n===== Métricas globales (media ± desviación estándar) =====")
print(f"Número de imágenes procesadas: {len(psnr_list)}")
print(f"PSNR medio    : {mean_psnr:.2f} ± {std_psnr:.2f} dB")
print(f"SSIM medio    : {mean_ssim:.4f} ± {std_ssim:.4f}")
print(f"MSE medio     : {mean_mse:.6f} ± {std_mse:.6f}")
print(f"FSIM medio    : {mean_fsim:.4f} ± {std_fsim:.4f}")
print(f"LPIPS medio   : {mean_lpips:.4f} ± {std_lpips:.4f}  (↓ mejor)")
print(f"NIQE medio    : {mean_niqe:.4f} ± {std_niqe:.4f}    (↓ mejor)")
print(f"BRISQUE medio : {mean_brisque:.4f} ± {std_brisque:.4f} (↓ mejor)")

summary_csv = "llie_resultados_globales.csv"

summary_row = {
    "Dataset": dataset_name,
    "Num_Images": len(psnr_list),
    "PSNR_mean": mean_psnr, "PSNR_std": std_psnr,
    "SSIM_mean": mean_ssim, "SSIM_std": std_ssim,
    "MSE_mean": mean_mse, "MSE_std": std_mse,
    "FSIM_mean": mean_fsim, "FSIM_std": std_fsim,
    "LPIPS_mean": mean_lpips, "LPIPS_std": std_lpips,
    "NIQE_mean": mean_niqe, "NIQE_std": std_niqe,
    "BRISQUE_mean": mean_brisque, "BRISQUE_std": std_brisque,
}

df_summary = pd.DataFrame([summary_row])

if not os.path.exists(summary_csv):
    df_summary.to_csv(summary_csv, index=False)
else:
    df_summary.to_csv(summary_csv, mode='a', header=False, index=False)

print(f"\nResumen global guardado/actualizado en: {summary_csv}")




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("llie_resultados_globales (copy).csv")

method_col = "Dataset"

candidate_metrics = [
    "PSNR_mean", "SSIM_mean", "FSIM_mean",
    "MSE_mean", "LPIPS_mean",
    "NIQE_mean", "BRISQUE_mean"
]
metrics = [m for m in candidate_metrics if m in df.columns]
if len(metrics) < 3:
    raise ValueError(f"Pocas métricas disponibles para radar. Encontré: {metrics}")

higher_better = {m: True for m in metrics}
for m in ["MSE_mean", "LPIPS_mean", "NIQE_mean", "BRISQUE_mean"]:
    if m in higher_better:
        higher_better[m] = False  # estas son (↓ mejor)

# Normalización por métrica (entre métodos)
vals = df[metrics].astype(float).copy()
norm = pd.DataFrame(index=df.index, columns=metrics, dtype=float)

for m in metrics:
    x = vals[m].to_numpy()
    mn, mx = np.nanmin(x), np.nanmax(x)
    if np.isclose(mx, mn):
        s = np.full_like(x, 0.5, dtype=float)
    else:
        if higher_better[m]:
            s = (x - mn) / (mx - mn)
        else:
            s = (mx - x) / (mx - mn)  # invierte (↓ mejor) -> (↑ mejor)
    norm[m] = s

# --- Radar plot ---
labels = [m.replace("_mean", "") for m in metrics]
N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0, 1)
ax.grid(True)

for i in norm.index:
    data = norm.loc[i, metrics].to_list()
    data += data[:1]
    ax.plot(angles, data, linewidth=2, label=str(df.loc[i, method_col]))
    ax.fill(angles, data, alpha=0.10)

ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.10))
plt.title("Radar por método (métricas normalizadas)")
plt.show()


In [ ]:
#%run src/LLIE_metrics.ipynb
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from piq import psnr, ssim, fsim
import lpips

from PIL import Image
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 0. Dispositivo
# ------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# ------------------------------------------------------------------
# 1. Nombre del dataset y rutas
# ------------------------------------------------------------------
dataset_name = "LOL Test"  # <-- cámbialo según el dataset

folder_low  = '../datasets/lol_dataset/eval15/low/'   # imágenes low-light (input del modelo)
folder_high = '../datasets/lol_dataset/eval15/high/'  # ground truth
out_dir     = "./results_eval15_enhanced_LOL"         # donde se guardarán las mejoradas
os.makedirs(out_dir, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

transform = transforms.ToTensor()

generator = Generator().to(device)
state_dict = torch.load(g_model_name, map_location=device)
generator.load_state_dict(state_dict)
generator.eval()


lpips_fn = lpips.LPIPS(net='alex').to(device).eval()


image_names = sorted([
    f for f in os.listdir(folder_low)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
])

print(f"Se encontraron {len(image_names)} imágenes low para evaluar.")


psnr_list  = []
ssim_list  = []
mse_list   = []
fsim_list  = []
lpips_list = []

for img_name in image_names:
    path_low  = os.path.join(folder_low, img_name)
    path_high = os.path.join(folder_high, img_name)
    print("low", path_low)
    print("high", path_high)

    if not os.path.exists(path_high):
        print(f"⚠️ No se encontró la GT para: {img_name}")
        continue

    # ---------------- Cargar imágenes como PIL ----------------
    low_pil  = Image.open(path_low).convert("RGB")
    high_pil = Image.open(path_high).convert("RGB")

    # ---------------- Aplicar transform ----------------
    input_tensor = transform(low_pil).unsqueeze(0).to(device)   # (1, C, H, W), [0,1]
    high_tensor  = transform(high_pil).unsqueeze(0).to(device)  # (1, C, H, W), [0,1]

    # ---------------- Inferencia ----------------
    with torch.no_grad():
        enhanced = generator(input_tensor)  # salida del modelo

    # Recortar a [0,1] por si el modelo se sale de rango
    enhanced = torch.clamp(enhanced, 0.0, 1.0)

    # ---------------- Métricas (PSNR/SSIM/FSIM/MSE) ----------------
    enhanced_for_metrics = enhanced.cpu()
    high_for_metrics     = high_tensor.cpu()
    data_range = 1.0

    psnr_value = psnr(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    ssim_value = ssim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    fsim_value = fsim(enhanced_for_metrics, high_for_metrics, data_range=data_range).item()
    mse_value  = F.mse_loss(enhanced_for_metrics, high_for_metrics).item()

    psnr_list.append(psnr_value)
    ssim_list.append(ssim_value)
    mse_list.append(mse_value)
    fsim_list.append(fsim_value)

    # ---------------- LPIPS ----------------
    # LPIPS requiere [-1,1]
    enhanced_lpips = enhanced * 2.0 - 1.0
    high_lpips     = high_tensor * 2.0 - 1.0

    with torch.no_grad():
        lpips_value = lpips_fn(enhanced_lpips, high_lpips).mean().item()

    lpips_list.append(lpips_value)

    # ---------------- Guardar salida visual ----------------
    save_path = os.path.join(out_dir, img_name)
    save_image(enhanced, save_path)


if len(psnr_list) == 0:
    raise RuntimeError("No se calcularon métricas: revisa que haya imágenes y GT correspondientes.")

psnr_arr  = np.array(psnr_list)
ssim_arr  = np.array(ssim_list)
mse_arr   = np.array(mse_list)
fsim_arr  = np.array(fsim_list)
lpips_arr = np.array(lpips_list)

mean_psnr = psnr_arr.mean()
std_psnr  = psnr_arr.std(ddof=1)

mean_ssim = ssim_arr.mean()
std_ssim  = ssim_arr.std(ddof=1)

mean_mse  = mse_arr.mean()
std_mse   = mse_arr.std(ddof=1)

mean_fsim = fsim_arr.mean()
std_fsim  = fsim_arr.std(ddof=1)

mean_lpips = lpips_arr.mean()
std_lpips  = lpips_arr.std(ddof=1)

print("\n===== Métricas globales (media ± desviación estándar) =====")
print(f"Número de imágenes procesadas: {len(psnr_list)}")
print(f"PSNR medio : {mean_psnr:.2f} ± {std_psnr:.2f} dB")
print(f"SSIM medio : {mean_ssim:.4f} ± {std_ssim:.4f}")
print(f"MSE medio  : {mean_mse:.6f} ± {std_mse:.6f}")
print(f"FSIM medio : {mean_fsim:.4f} ± {std_fsim:.4f}")
print(f"LPIPS medio: {mean_lpips:.4f} ± {std_lpips:.4f}  (↓ mejor)")

# ------------------------------------------------------------------
# 7. Guardar resumen global en un CSV acumulativo
# ------------------------------------------------------------------
summary_csv = "llie_resultados_globales.csv"  # nombre del CSV global

num_imgs = len(psnr_list)

summary_row = {
    "Dataset": dataset_name,
    "Num_Images": num_imgs,
    "PSNR_mean": mean_psnr,
    "PSNR_std": std_psnr,
    "SSIM_mean": mean_ssim,
    "SSIM_std": std_ssim,
    "MSE_mean": mean_mse,
    "MSE_std": std_mse,
    "FSIM_mean": mean_fsim,
    "FSIM_std": std_fsim,
    "LPIPS_mean": mean_lpips,
    "LPIPS_std": std_lpips,
}

df_summary = pd.DataFrame([summary_row])

# Crear o append al CSV sin borrar lo anterior
if not os.path.exists(summary_csv):
    df_summary.to_csv(summary_csv, index=False)
else:
    df_summary.to_csv(summary_csv, mode='a', header=False, index=False)

print(f"\nResumen global guardado/actualizado en: {summary_csv}")



In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from piq import fsim   # FSIM lo usamos como métrica extra
from PIL import Image
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)
%run src/LLIE_metrics.ipynb


dataset_name = "LOL-v2 Real Test" 

folder_low  = '../datasets/LOL-v2/Real_captured/Test/Low/'     # imágenes low-light
folder_high = '../datasets/LOL-v2/Real_captured/Test/Normal/'  # ground truth
out_dir     = "./results_eval15_enhanced_LOL-v2-r"             # donde se guardarán las mejoradas
os.makedirs(out_dir, exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# ------------------------------------------------------------------
# 2. Transform (ToTensor -> [0,1])
# ------------------------------------------------------------------
transform = transforms.ToTensor()

# ------------------------------------------------------------------
# 3. Cargar modelo
# ------------------------------------------------------------------
generator = Generator().to(device)
state_dict = torch.load(g_model_name, map_location=device)
generator.load_state_dict(state_dict)
generator.eval()
print("Modelo cargado desde:", g_model_name)


evaluator = ImageQualityEvaluator(
    device=device,
    data_range=1.0,
    use_pyiqa_niqe=False
)

evaluator.add_metric(
    "FSIM",
    lambda low, enh, high: fsim(enh, high, data_range=1.0) if high is not None else float('nan')
)

evaluator.add_metric(
    "MSE",
    lambda low, enh, high: F.mse_loss(enh, high) if high is not None else float('nan')
)


image_names = sorted([
    f for f in os.listdir(folder_low)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
])

print(f"Se encontraron {len(image_names)} imágenes low para evaluar.")

# ------------------------------------------------------------------
# 5. Loop: inferencia + métricas
# ------------------------------------------------------------------
psnr_list      = []
ssim_list      = []
mse_list       = []
fsim_list      = []
loe_list       = []
niqe_enh_list  = []
niqe_low_list  = []
lpips_list     = []
vif_list       = []

for img_name in image_names:
    path_low  = os.path.join(folder_low, img_name)
    path_high = os.path.join(folder_high, img_name)

    if not os.path.exists(path_high):
        print(f"⚠️ No se encontró la GT para: {img_name}")
        continue

    # Cargar imágenes
    low_pil  = Image.open(path_low).convert("RGB")
    high_pil = Image.open(path_high).convert("RGB")

    # A tensores [1,C,H,W] en [0,1]
    input_tensor = transform(low_pil).unsqueeze(0).to(device)
    high_tensor  = transform(high_pil).unsqueeze(0).to(device)

    # Inferencia
    with torch.no_grad():
        enhanced = generator(input_tensor)

    # Asegurar rango [0,1]
    enhanced = torch.clamp(enhanced, 0.0, 1.0)

    low_for_metrics      = input_tensor
    enhanced_for_metrics = enhanced
    high_for_metrics     = high_tensor

    # =================== MÉTRICAS AQUÍ ===================
    metrics = evaluator(low_for_metrics, enhanced_for_metrics, high_for_metrics)
    # metrics es un dict con las métricas registradas en la clase


    psnr_list.append(metrics.get("PSNR", float('nan')))
    ssim_list.append(metrics.get("SSIM", float('nan')))
    mse_list.append(metrics.get("MSE", float('nan')))
    fsim_list.append(metrics.get("FSIM", float('nan')))
    loe_list.append(metrics.get("LOE", float('nan')))
    niqe_enh_list.append(metrics.get("NIQE_enh", float('nan')))
    niqe_low_list.append(metrics.get("NIQE_low", float('nan')))
    lpips_list.append(metrics.get("LPIPS", float('nan')))
    vif_list.append(metrics.get("VIF", float('nan')))

    # Guardar salida visual
    save_path = os.path.join(out_dir, img_name)
    save_image(enhanced, save_path)


if len(psnr_list) == 0:
    raise RuntimeError("No se calcularon métricas: revisa que haya imágenes y GT correspondientes.")

def mean_std(arr):
    arr = np.array(arr, dtype=np.float64)
    return arr.mean(), arr.std(ddof=1)

mean_psnr,      std_psnr      = mean_std(psnr_list)
mean_ssim,      std_ssim      = mean_std(ssim_list)
mean_mse,       std_mse       = mean_std(mse_list)
mean_fsim,      std_fsim      = mean_std(fsim_list)
mean_loe,       std_loe       = mean_std(loe_list)
mean_niqe_enh,  std_niqe_enh  = mean_std(niqe_enh_list)
mean_niqe_low,  std_niqe_low  = mean_std(niqe_low_list)
mean_lpips,     std_lpips     = mean_std(lpips_list)
mean_vif,       std_vif       = mean_std(vif_list)

print("\n===== Métricas globales (media ± desviación estándar) =====")
print(f"Número de imágenes procesadas: {len(psnr_list)}")
print(f"PSNR medio       : {mean_psnr:.2f} ± {std_psnr:.2f} dB")
print(f"SSIM medio       : {mean_ssim:.4f} ± {std_ssim:.4f}")
print(f"MSE medio        : {mean_mse:.6f} ± {std_mse:.6f}")
print(f"FSIM medio       : {mean_fsim:.4f} ± {std_fsim:.4f}")
print(f"LOE medio        : {mean_loe:.4f} ± {std_loe:.4f}")
print(f"NIQE_enh medio   : {mean_niqe_enh:.4f} ± {std_niqe_enh:.4f}")
print(f"NIQE_low medio   : {mean_niqe_low:.4f} ± {std_niqe_low:.4f}")
print(f"LPIPS medio      : {mean_lpips:.4f} ± {std_lpips:.4f}")
print(f"VIF medio        : {mean_vif:.44f} ± {std_vif:.4f}")


summary_csv = "llie_resultados_globales.csv"
num_imgs = len(psnr_list)

summary_row = {
    "Dataset": dataset_name,
    "Num_Images": num_imgs,
    "PSNR_mean": mean_psnr,
    "PSNR_std": std_psnr,
    "SSIM_mean": mean_ssim,
    "SSIM_std": std_ssim,
    "MSE_mean": mean_mse,
    "MSE_std": std_mse,
    "FSIM_mean": mean_fsim,
    "FSIM_std": std_fsim,
    "LOE_mean": mean_loe,
    "LOE_std": std_loe,
    "NIQE_enh_mean": mean_niqe_enh,
    "NIQE_enh_std": std_niqe_enh,
    "NIQE_low_mean": mean_niqe_low,
    "NIQE_low_std": std_niqe_low,
    "LPIPS_mean": mean_lpips,
    "LPIPS_std": std_lpips,
    "VIF_mean": mean_vif,
    "VIF_std": std_vif,
}

df_summary = pd.DataFrame([summary_row])

if not os.path.exists(summary_csv):
    df_summary.to_csv(summary_csv, index=False)
else:
    df_summary.to_csv(summary_csv, mode='a', header=False, index=False)

print(f"\nResumen global guardado/actualizado en: {summary_csv}")


In [ ]:
from PIL import Image

generator = Generator().to(device)
generator.load_state_dict(torch.load(g_model_name))
generator.eval()  # Ponemos el modelo en modo de inferencia
img_test_path      = '../datasets/lol_dataset/eval15/low/1.png'#'/content/drive/MyDrive/datasets/lol_dataset/eval15/low/22.png'
img_high_test_path ='../datasets/lol_dataset/eval15/high/1.png'#'/content/drive/MyDrive/datasets/lol_dataset/eval15/high/22.png'
# Cargar y preparar la imagen de baja luminosidad (low-light)
high_image  = Image.open(img_high_test_path)
high_image  = transform(high_image).unsqueeze(0).to(device)
input_image = Image.open(img_test_path)  # Cambia esta ruta a la de tu imagen de baja luminosidad
input_image = transform(input_image).unsqueeze(0).to(device)  # Aplica las transformaciones y agrega una dimensión de batch

# Realizar inferencia para mejorar la imagen de baja luminosidad
with torch.no_grad():  # Desactiva el cálculo de gradientes para la inferencia
    enhanced_image = generator(input_image)  # Genera la imagen mejorada

# Guardar o mostrar la imagen mejorada
save_image(enhanced_image, "resultado_inferencia.png")

# Si quieres visualizar la imagen:
import matplotlib.pyplot as plt
plt.subplot(1, 3, 1),plt.imshow(input_image.cpu().squeeze(0).permute(1, 2, 0).numpy()), plt.title('Input image'), plt.axis('off')
plt.subplot(1, 3, 2),plt.imshow(enhanced_image.cpu().squeeze(0).permute(1, 2, 0).numpy()), plt.title('Enhanced image'),plt.axis('off')
plt.subplot(1, 3, 3),plt.imshow(high_image.cpu().squeeze(0).permute(1, 2, 0).numpy()), plt.title('Original image'),plt.axis('off')
plt.savefig('salida.png', dpi=300, bbox_inches='tight')  # Guarda la imagen en alta resolución

plt.show()

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.io import read_image
import torch.nn.functional as F
from piq import psnr, ssim, fsim
import os
import pandas as pd

# Rutas de las carpetas
folder_high= '../datasets/lol_dataset/eval15/high/'
folder_low      = '../datasets/lol_dataset/eval15/low/'

# Transformación: Convertir a tensor y normalizar a [0,1]
def load_image(image_path):
    image = read_image(image_path).float() / 255.0  # Cargar imagen y normalizar
    return image.unsqueeze(0)  # Añadir dimensión batch

# Obtener nombres de archivos en la carpeta de baja iluminación
image_names = sorted(os.listdir(folder_low))

# Lista para almacenar métricas
metrics = []

# Iterar sobre cada imagen
for image_name in image_names:
    path_low = os.path.join(folder_low, image_name)
    path_high = os.path.join(folder_high, image_name)  # Se asume el mismo nombre

    # Verificar si la imagen de referencia existe
    if not os.path.exists(path_high):
        print(f"⚠️ Imagen de referencia no encontrada: {image_name}")
        continue

    # Cargar imágenes
    img_low = load_image(path_low)
    img_high = load_image(path_high)

    # Asegurar que las imágenes tengan el mismo tamaño
    assert img_low.shape == img_high.shape, f"Las imágenes deben tener el mismo tamaño: {image_name}"

    # 📌 MÉTRICAS
    psnr_value = psnr(img_low, img_high, data_range=1.0).item()
    ssim_value = ssim(img_low, img_high, data_range=1.0).item()
    mse_value = F.mse_loss(img_low, img_high).item()
    fsim_value = fsim(img_low, img_high, data_range=1.0).item()

    # Guardar en lista
    metrics.append([image_name, psnr_value, ssim_value, mse_value, fsim_value])

# Convertir a DataFrame y mostrar
df_metrics = pd.DataFrame(metrics, columns=["Imagen", "PSNR", "SSIM", "MSE", "FSIM"])
# Mostrar el DataFrame en Jupyter Notebook o terminal
print(df_metrics)  # Para terminal
df_metrics  # Para Jupyter Notebook
#import ace_tools as tools
#tools.display_dataframe_to_user(name="Métricas de Low-Light Enhancement", dataframe=df_metrics)